# The v1 database in DuckDB SQL

What this notebook shows, with SQL only:
1. Which tables are in `data/cropyield.duckdb`, and how many rows each has.
2. The columns and the first rows of every table and view.
3. The months of each country and cohort, split by fold (`s1_cc2km`).

**Rules**
- The database is opened `read_only=True`. This notebook cannot change the team file (it is pinned in `configs/team_database_v1.json`).
- The folds are not in the database. They are in the frozen split files `data/splits/<cohort>__s1_cc2km.csv`. DuckDB reads the CSV files directly, and the join is on `obs_id`.
- Fold 0 of `s1_cc2km` is kept for the end of the project (owner ruling, 1 Nov). It is shown here, but no new model is scored on it.
- Run it from `notebooks/` with the `.venv` kernel of the repository.

In [1]:
from pathlib import Path

import duckdb
import pandas as pd

REPO = Path.cwd().parent                           # this notebook lives in notebooks/
DB = REPO / "data" / "cropyield.duckdb"            # v1 team database
SPLITS = (REPO / "data" / "splits").as_posix()      # the frozen split CSV files

con = duckdb.connect(str(DB), read_only=True)      # read_only: no query here can change the file
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 50)


def sql(query):
    """Run one SQL query and return the result as a pandas table."""
    return con.execute(query).df()


def show(table, n=3):
    """Print the columns (name and type) and the first n rows of one table or view."""
    print(f"=== {table} ===")
    display(sql(f"SELECT column_name, column_type FROM (DESCRIBE {table})"))
    display(sql(f"SELECT * FROM {table} LIMIT {n}"))


print("open:", DB)

open: C:\Dev\DSC-291-Final-Project\data\cropyield.duckdb


## 1. The tables and their row counts

- **Tables** hold the data. **Views** are saved queries over the tables (`master` joins everything into one row per field).
- `obs_id` = `dataset:source_id` is the key in every table.

In [2]:
sql("SELECT table_name, table_type FROM information_schema.tables ORDER BY table_type, table_name")

,table_name,table_type
0,build_info,BASE TABLE
1,coverage,BASE TABLE
2,downloads,BASE TABLE
3,labels,BASE TABLE
4,monthly_bands,BASE TABLE
5,mosaiks_monthly,BASE TABLE
6,mosaiks_season,BASE TABLE
7,presto_embeddings,BASE TABLE
8,scenes,BASE TABLE
9,vegetation_indices,BASE TABLE


In [3]:
tables = sql("SELECT table_name FROM information_schema.tables WHERE table_type = 'BASE TABLE' ORDER BY 1").table_name
pd.DataFrame({"rows": {t: con.execute(f"SELECT count(*) FROM {t}").fetchone()[0] for t in tables}})

,rows
build_info,1
coverage,98299
downloads,11364
labels,11364
monthly_bands,98299
mosaiks_monthly,84967
mosaiks_season,11364
presto_embeddings,11364
scenes,84967
vegetation_indices,98299


## 2. Columns and first rows of each table

| Table | One row is |
|---|---|
| `labels` | one field: yield, position, season, include or exclude |
| `downloads` | one field: the photo download result |
| `scenes` | one accepted photo of one field (one per month at most) |
| `coverage` | one field and one month: `clear`, `cloudy` or `no_scene` |
| `monthly_bands` | one field and one month: the median of the 10 bands over the field pixels |
| `vegetation_indices` | one field and one month: the 5 green indices |
| `presto_embeddings` | one field: 128 Presto numbers |
| `mosaiks_season` | one field: 512 MOSAIKS numbers for the chip + 512 for the field only |
| `mosaiks_monthly` | one field and one month: the same, per month |

In [4]:
show("labels")

=== labels ===


,column_name,column_type
0,obs_id,VARCHAR
1,cohort,VARCHAR
2,dataset,VARCHAR
3,source_id,VARCHAR
4,source_row,INTEGER
5,country,VARCHAR
6,year,INTEGER
7,yield_kg_ha,DOUBLE
8,source_yield,DOUBLE
9,source_yield_units,VARCHAR


,obs_id,cohort,dataset,source_id,source_row,country,year,yield_kg_ha,source_yield,source_yield_units,yield_field,latitude,longitude,geometry_type,field_area_ha,label_date,season_start,season_end,intercropping,field_polygon,include_primary,exclusion_reason
0,malawi_2022:3ee4d3a6-3a6e-4dae-9046-66009b51d884,malawi_2022,malawi_2022,3ee4d3a6-3a6e-4dae-9046-66009b51d884,1,malawi,2022,3731.0,3.731,t/ha,AVG_YLD,-15.052553,35.204831,Polygon,0.133246,2022-04-30,2021-10-01,2022-06-30,None,"{""type"": ""Polygon"", ""coordinates"": [[[35.20461...",False,dropped_cohort_2022_imagery_gap
1,malawi_2022:4f780999-86d8-42b6-bc2a-3f45a6c56a63,malawi_2022,malawi_2022,4f780999-86d8-42b6-bc2a-3f45a6c56a63,2,malawi,2022,2488.0,2.488,t/ha,AVG_YLD,-15.046088,34.905687,Polygon,0.312441,2022-05-02,2021-10-01,2022-06-30,None,"{""type"": ""Polygon"", ""coordinates"": [[[34.90594...",False,dropped_cohort_2022_imagery_gap
2,malawi_2022:9d9a8fb6-2b21-4d32-b9de-43f0d00af118,malawi_2022,malawi_2022,9d9a8fb6-2b21-4d32-b9de-43f0d00af118,3,malawi,2022,1224.0,1.224,t/ha,AVG_YLD,-15.091802,34.975304,Polygon,0.620347,2022-05-02,2021-10-01,2022-06-30,None,"{""type"": ""Polygon"", ""coordinates"": [[[34.97540...",False,dropped_cohort_2022_imagery_gap


In [5]:
show("downloads")

=== downloads ===


,column_name,column_type
0,obs_id,VARCHAR
1,record_dir,VARCHAR
2,status,VARCHAR
3,aoi_type,VARCHAR
4,aoi_pixels,INTEGER
5,field_pixels,INTEGER
6,aoi_inside_chip,BOOLEAN
7,catalog_scenes,INTEGER
8,accepted_months,INTEGER
9,crs,VARCHAR


,obs_id,record_dir,status,aoi_type,aoi_pixels,field_pixels,aoi_inside_chip,catalog_scenes,accepted_months,crs,search_start,search_end
0,malawi_2022:3ee4d3a6-3a6e-4dae-9046-66009b51d884,data/imagery/sentinel2/run-517bfcf0a69c6cac/re...,complete,field_polygon,26,11,True,16,3,EPSG:32736,2021-10-01,2022-06-30
1,malawi_2022:4f780999-86d8-42b6-bc2a-3f45a6c56a63,data/imagery/sentinel2/run-517bfcf0a69c6cac/re...,complete,field_polygon,47,32,True,32,3,EPSG:32736,2021-10-01,2022-06-30
2,malawi_2022:9d9a8fb6-2b21-4d32-b9de-43f0d00af118,data/imagery/sentinel2/run-517bfcf0a69c6cac/re...,complete,field_polygon,82,64,True,16,3,EPSG:32736,2021-10-01,2022-06-30


In [6]:
show("scenes")

=== scenes ===


,column_name,column_type
0,obs_id,VARCHAR
1,item_id,VARCHAR
2,acquired,DATE
3,month,VARCHAR
4,satellite,VARCHAR
5,tile,VARCHAR
6,tile_cloud_cover,DOUBLE
7,clear_fraction_aoi,DOUBLE
8,field_valid_pixels,INTEGER


,obs_id,item_id,acquired,month,satellite,tile,tile_cloud_cover,clear_fraction_aoi,field_valid_pixels
0,malawi_2022:3ee4d3a6-3a6e-4dae-9046-66009b51d884,S2B_T36LYJ_20211010T080248_L2A,2021-10-10,2021-10,S2B,36LYJ,0.033044,1.0,11
1,malawi_2022:3ee4d3a6-3a6e-4dae-9046-66009b51d884,S2A_T36LYJ_20211124T080103_L2A,2021-11-24,2021-11,S2A,36LYJ,1.158776,1.0,11
2,malawi_2022:3ee4d3a6-3a6e-4dae-9046-66009b51d884,S2A_T36LYJ_20211224T080158_L2A,2021-12-24,2021-12,S2A,36LYJ,9.997949,1.0,11


In [7]:
show("coverage")

=== coverage ===


,column_name,column_type
0,obs_id,VARCHAR
1,month,VARCHAR
2,state,VARCHAR
3,item_id,VARCHAR


,obs_id,month,state,item_id
0,malawi_2022:3ee4d3a6-3a6e-4dae-9046-66009b51d884,2021-10,clear,S2B_T36LYJ_20211010T080248_L2A
1,malawi_2022:3ee4d3a6-3a6e-4dae-9046-66009b51d884,2021-11,clear,S2A_T36LYJ_20211124T080103_L2A
2,malawi_2022:3ee4d3a6-3a6e-4dae-9046-66009b51d884,2021-12,clear,S2A_T36LYJ_20211224T080158_L2A


In [8]:
show("monthly_bands")

=== monthly_bands ===


,column_name,column_type
0,obs_id,VARCHAR
1,month,VARCHAR
2,b02,DOUBLE
3,b03,DOUBLE
4,b04,DOUBLE
5,b05,DOUBLE
6,b06,DOUBLE
7,b07,DOUBLE
8,b08,DOUBLE
9,b8a,DOUBLE


,obs_id,month,b02,b03,b04,b05,b06,b07,b08,b8a,b11,b12,field_pixels,missing
0,malawi_2022:3ee4d3a6-3a6e-4dae-9046-66009b51d884,2021-10,0.0714,0.0939,0.1278,0.155388,0.190056,0.211175,0.2194,0.238912,0.357019,0.333644,11,False
1,malawi_2022:3ee4d3a6-3a6e-4dae-9046-66009b51d884,2021-11,0.0714,0.0936,0.1198,0.156113,0.198194,0.216981,0.2218,0.245481,0.343419,0.311156,11,False
2,malawi_2022:3ee4d3a6-3a6e-4dae-9046-66009b51d884,2021-12,0.0670,0.0962,0.1172,0.150281,0.196244,0.210338,0.2150,0.235256,0.319450,0.290050,11,False


In [9]:
show("vegetation_indices")

=== vegetation_indices ===


,column_name,column_type
0,obs_id,VARCHAR
1,month,VARCHAR
2,ndvi,DOUBLE
3,gcvi,DOUBLE
4,mtci,DOUBLE
5,ndvi705,DOUBLE
6,ndvi740,DOUBLE


,obs_id,month,ndvi,gcvi,mtci,ndvi705,ndvi740
0,malawi_2022:3ee4d3a6-3a6e-4dae-9046-66009b51d884,2021-10,0.263825,1.336528,2.320344,0.170797,0.071665
1,malawi_2022:3ee4d3a6-3a6e-4dae-9046-66009b51d884,2021-11,0.298595,1.369658,1.808950,0.173817,0.056206
2,malawi_2022:3ee4d3a6-3a6e-4dae-9046-66009b51d884,2021-12,0.294401,1.234927,1.956358,0.177175,0.045609


The Presto and MOSAIKS columns are long lists of numbers. So the next cells show the length of each list and its first 5 numbers.

In [10]:
display(sql("SELECT column_name, column_type FROM (DESCRIBE presto_embeddings)"))
sql("SELECT obs_id, len(embedding) AS n_numbers, embedding[1:5] AS first_5 FROM presto_embeddings LIMIT 3")

,column_name,column_type
0,obs_id,VARCHAR
1,embedding,FLOAT[128]


,obs_id,n_numbers,first_5
0,malawi_2022:3ee4d3a6-3a6e-4dae-9046-66009b51d884,128,"[-1.6785579, -5.2099504, 1.8612437, -0.8976414..."
1,malawi_2022:4f780999-86d8-42b6-bc2a-3f45a6c56a63,128,"[-1.4031829, -5.0533423, 1.8900476, -1.0976031..."
2,malawi_2022:9d9a8fb6-2b21-4d32-b9de-43f0d00af118,128,"[-1.3516289, -4.930976, 1.7934849, -1.240423, ..."


In [11]:
display(sql("SELECT column_name, column_type FROM (DESCRIBE mosaiks_season)"))
sql("""
SELECT obs_id, months_used,
       len(chip)  AS n_chip,  chip[1:5]  AS chip_first_5,
       len(field) AS n_field, field[1:5] AS field_first_5
FROM mosaiks_season LIMIT 3
""")

,column_name,column_type
0,obs_id,VARCHAR
1,months_used,INTEGER
2,chip,FLOAT[]
3,field,FLOAT[]


,obs_id,months_used,n_chip,chip_first_5,n_field,field_first_5
0,malawi_2022:3ee4d3a6-3a6e-4dae-9046-66009b51d884,3,1024,"[0.0143349795, 7.4279084, 0.002995304, 5.50481...",1024,"[0.0, 7.2768803, 0.0, 8.558207, 0.0]"
1,malawi_2022:4f780999-86d8-42b6-bc2a-3f45a6c56a63,3,1024,"[0.0, 9.972781, 0.0, 8.366978, 0.0]",1024,"[0.0, 9.907872, 0.0, 6.9652066, 0.0]"
2,malawi_2022:9d9a8fb6-2b21-4d32-b9de-43f0d00af118,3,1024,"[0.0, 10.5020485, 0.0, 0.7110631, 0.0]",1024,"[0.0, 9.480282, 0.0, 1.5737518, 0.0]"


In [12]:
display(sql("SELECT column_name, column_type FROM (DESCRIBE mosaiks_monthly)"))
sql("SELECT obs_id, month, len(chip) AS n_chip, chip[1:5] AS chip_first_5 FROM mosaiks_monthly LIMIT 3")

,column_name,column_type
0,obs_id,VARCHAR
1,month,VARCHAR
2,chip,FLOAT[]
3,field,FLOAT[]


,obs_id,month,n_chip,chip_first_5
0,malawi_2022:3ee4d3a6-3a6e-4dae-9046-66009b51d884,2021-10,1024,"[3.094619e-05, 7.4652543, 0.00010765591, 4.588..."
1,malawi_2022:3ee4d3a6-3a6e-4dae-9046-66009b51d884,2021-11,1024,"[0.010791022, 7.356375, 0.004348833, 6.5044093..."
2,malawi_2022:3ee4d3a6-3a6e-4dae-9046-66009b51d884,2021-12,1024,"[0.03218297, 7.4620967, 0.0045294235, 5.421435..."


### The views

- `master`: one row per field, with the labels, the download status, `months_with_data`, the season indices, Presto and MOSAIKS.
- `monthly`: one row per field and month (bands + indices together).
- `season_indices`: one row per field, with the max and the mean of each index over the season.

In [13]:
show("master", n=2)

=== master ===


,column_name,column_type
0,obs_id,VARCHAR
1,cohort,VARCHAR
2,dataset,VARCHAR
3,source_id,VARCHAR
4,source_row,INTEGER
5,country,VARCHAR
6,year,INTEGER
7,yield_kg_ha,DOUBLE
8,source_yield,DOUBLE
9,source_yield_units,VARCHAR


,obs_id,cohort,dataset,source_id,source_row,country,year,yield_kg_ha,source_yield,source_yield_units,yield_field,latitude,longitude,geometry_type,field_area_ha,label_date,season_start,season_end,intercropping,field_polygon,include_primary,exclusion_reason,has_imagery,download_status,months_with_data,ndvi_season_max,ndvi_season_mean,gcvi_season_max,gcvi_season_mean,mtci_season_max,mtci_season_mean,ndvi705_season_max,ndvi705_season_mean,ndvi740_season_max,ndvi740_season_mean,presto_embedding,mosaiks_chip,mosaiks_field,mosaiks_months_used
0,malawi_2022:3ee4d3a6-3a6e-4dae-9046-66009b51d884,malawi_2022,malawi_2022,3ee4d3a6-3a6e-4dae-9046-66009b51d884,1,malawi,2022,3731.0,3.731,t/ha,AVG_YLD,-15.052553,35.204831,Polygon,0.133246,2022-04-30,2021-10-01,2022-06-30,None,"{""type"": ""Polygon"", ""coordinates"": [[[35.20461...",False,dropped_cohort_2022_imagery_gap,True,complete,3,0.298595,0.285607,1.369658,1.313705,2.320344,2.028550,0.177175,0.173930,0.071665,0.057827,"[-1.6785579, -5.2099504, 1.8612437, -0.8976414...","[0.0143349795, 7.4279084, 0.002995304, 5.50481...","[0.0, 7.2768803, 0.0, 8.558207, 0.0, 0.0, 0.52...",3
1,malawi_2022:4f780999-86d8-42b6-bc2a-3f45a6c56a63,malawi_2022,malawi_2022,4f780999-86d8-42b6-bc2a-3f45a6c56a63,2,malawi,2022,2488.0,2.488,t/ha,AVG_YLD,-15.046088,34.905687,Polygon,0.312441,2022-05-02,2021-10-01,2022-06-30,None,"{""type"": ""Polygon"", ""coordinates"": [[[34.90594...",False,dropped_cohort_2022_imagery_gap,True,complete,3,0.221357,0.217163,1.215943,1.162016,2.492184,2.240539,0.148554,0.140099,0.097987,0.091995,"[-1.4031829, -5.0533423, 1.8900476, -1.0976031...","[0.0, 9.972781, 0.0, 8.366978, 0.0, 0.0, 0.638...","[0.0, 9.907872, 0.0, 6.9652066, 0.0, 0.0, 0.04...",3


In [14]:
show("monthly")

=== monthly ===


,column_name,column_type
0,obs_id,VARCHAR
1,cohort,VARCHAR
2,month,VARCHAR
3,coverage_state,VARCHAR
4,b02,DOUBLE
5,b03,DOUBLE
6,b04,DOUBLE
7,b05,DOUBLE
8,b06,DOUBLE
9,b07,DOUBLE


,obs_id,cohort,month,coverage_state,b02,b03,b04,b05,b06,b07,b08,b8a,b11,b12,ndvi,gcvi,mtci,ndvi705,ndvi740,mosaiks_chip,mosaiks_field,yield_kg_ha
0,malawi_2022:3ee4d3a6-3a6e-4dae-9046-66009b51d884,malawi_2022,2021-10,clear,0.0714,0.0939,0.1278,0.155388,0.190056,0.211175,0.2194,0.238912,0.357019,0.333644,0.263825,1.336528,2.320344,0.170797,0.071665,"[3.094619e-05, 7.4652543, 0.00010765591, 4.588...","[0.0, 7.5486693, 0.0, 8.497971, 0.0, 0.0, 1.38...",3731.0
1,malawi_2022:3ee4d3a6-3a6e-4dae-9046-66009b51d884,malawi_2022,2021-11,clear,0.0714,0.0936,0.1198,0.156113,0.198194,0.216981,0.2218,0.245481,0.343419,0.311156,0.298595,1.369658,1.808950,0.173817,0.056206,"[0.010791022, 7.356375, 0.004348833, 6.5044093...","[0.0, 7.073415, 0.0, 9.446264, 0.0, 0.0, 0.196...",3731.0
2,malawi_2022:3ee4d3a6-3a6e-4dae-9046-66009b51d884,malawi_2022,2021-12,clear,0.0670,0.0962,0.1172,0.150281,0.196244,0.210338,0.2150,0.235256,0.319450,0.290050,0.294401,1.234927,1.956358,0.177175,0.045609,"[0.03218297, 7.4620967, 0.0045294235, 5.421435...","[0.0, 7.2085576, 0.0, 7.730385, 0.0, 0.0, 0.0,...",3731.0


In [15]:
show("season_indices")

=== season_indices ===


,column_name,column_type
0,obs_id,VARCHAR
1,ndvi_season_max,DOUBLE
2,ndvi_season_mean,DOUBLE
3,gcvi_season_max,DOUBLE
4,gcvi_season_mean,DOUBLE
5,mtci_season_max,DOUBLE
6,mtci_season_mean,DOUBLE
7,ndvi705_season_max,DOUBLE
8,ndvi705_season_mean,DOUBLE
9,ndvi740_season_max,DOUBLE


,obs_id,ndvi_season_max,ndvi_season_mean,gcvi_season_max,gcvi_season_mean,mtci_season_max,mtci_season_mean,ndvi705_season_max,ndvi705_season_mean,ndvi740_season_max,ndvi740_season_mean
0,malawi_2022:9d9a8fb6-2b21-4d32-b9de-43f0d00af118,0.225632,0.191163,1.146470,0.937620,2.344481,1.860163,0.148172,0.116464,0.091731,0.064390
1,malawi_2022:e258405b-20a3-4fd1-8667-164e6917717a,0.380222,0.289014,1.736591,1.414101,2.195453,2.165868,0.233455,0.180650,0.085580,0.075084
2,malawi_2022:db312764-e21c-42be-bcb9-a18337a2de9d,0.331081,0.301092,1.571802,1.391622,3.360400,2.573442,0.237146,0.198487,0.083968,0.073185


## 3. Months per country and fold

`SPLIT` below is not a table in the database. It is the text of a DuckDB call that reads the 4 frozen `s1_cc2km` CSV files.
Each split file holds the model fields of one cohort (`include_primary` true), with their `fold` (0–4) and `group_id`.

Malawi 2022 is in `labels` but excluded in v1, so it has no split file here.

In [16]:
SPLIT = (f"read_csv('{SPLITS}/*__s1_cc2km.csv', union_by_name = true, "
         "types = {'obs_id': 'VARCHAR', 'group_id': 'VARCHAR', 'fold': 'INTEGER'})")
sql(f"SELECT * FROM {SPLIT} LIMIT 3")

,obs_id,cohort,season_year,cluster_id,group_id,fold,repeat,family
0,one_acre_fund_full:ee151220,kenya_oaf_2019_2020,2019,0,0,2,0,s1_cc2km
1,one_acre_fund_full:454485a8,kenya_oaf_2019_2020,2019,1,1,4,0,s1_cc2km
2,one_acre_fund_full:37b3df46,kenya_oaf_2019_2020,2019,0,0,2,0,s1_cc2km


### 3a. The season window of each cohort

`season_start` and `season_end` come from the labels. `coverage` has one row for every month in that window.
Kenya has two seasons (2019 and 2020, February to September each), so its 16 months have a gap from October to January.

In [17]:
sql("""
SELECT country, cohort, season_start, season_end, count(*) AS model_fields
FROM labels
WHERE include_primary
GROUP BY ALL
ORDER BY country, cohort, season_start
""")

,country,cohort,season_start,season_end,model_fields
0,kenya,kenya_oaf_2019_2020,2019-02-01,2019-09-30,1768
1,kenya,kenya_oaf_2019_2020,2020-02-01,2020-09-30,2209
2,malawi,malawi_2023,2022-10-01,2023-06-30,260
3,zimbabwe,zimbabwe_2024,2023-10-01,2024-06-30,2458
4,zimbabwe,zimbabwe_lacuna_2022_2023,2022-10-01,2023-06-30,1880


In [18]:
months = sql("""
SELECT l.country, l.cohort,
       count(DISTINCT c.month)                      AS n_months,
       array_agg(DISTINCT c.month ORDER BY c.month) AS months
FROM coverage c
JOIN labels l USING (obs_id)
WHERE l.include_primary
GROUP BY ALL
ORDER BY 1, 2
""")
for row in months.itertuples():
    print(f"{row.cohort:27s} {row.n_months:2d} months: {', '.join(row.months)}")

kenya_oaf_2019_2020         16 months: 2019-02, 2019-03, 2019-04, 2019-05, 2019-06, 2019-07, 2019-08, 2019-09, 2020-02, 2020-03, 2020-04, 2020-05, 2020-06, 2020-07, 2020-08, 2020-09
malawi_2023                  9 months: 2022-10, 2022-11, 2022-12, 2023-01, 2023-02, 2023-03, 2023-04, 2023-05, 2023-06
zimbabwe_2024                9 months: 2023-10, 2023-11, 2023-12, 2024-01, 2024-02, 2024-03, 2024-04, 2024-05, 2024-06
zimbabwe_lacuna_2022_2023    9 months: 2022-10, 2022-11, 2022-12, 2023-01, 2023-02, 2023-03, 2023-04, 2023-05, 2023-06


### 3b. Each fold: fields, groups, clear months

- `fields`: the fields of the fold. `groups`: the 2 km clusters in it (a cluster is never split over two folds).
- `avg_clear_months`: the mean number of months with a clear photo (`months_with_data` in `master`).
- `model_rows`: the fields with at least 4 clear months. Only these go into the model (`load_xy`).

In [19]:
sql(f"""
SELECT s.cohort, s.fold,
       count(*)                                        AS fields,
       count(DISTINCT s.group_id)                      AS groups,
       round(avg(m.months_with_data), 2)               AS avg_clear_months,
       count(*) FILTER (WHERE m.months_with_data >= 4) AS model_rows,
       round(avg(m.yield_kg_ha) / 1000, 2)             AS mean_yield_t_ha
FROM {SPLIT} s
JOIN master m USING (obs_id)
GROUP BY ALL
ORDER BY s.cohort, s.fold
""")

,cohort,fold,fields,groups,avg_clear_months,model_rows,mean_yield_t_ha
0,kenya_oaf_2019_2020,0,796,169,7.38,796,3.11
1,kenya_oaf_2019_2020,1,796,170,7.40,796,3.06
2,kenya_oaf_2019_2020,2,795,170,7.42,795,3.01
3,kenya_oaf_2019_2020,3,795,170,7.41,795,3.31
4,kenya_oaf_2019_2020,4,795,170,7.46,795,3.21
5,malawi_2023,0,52,9,4.92,49,2.62
6,malawi_2023,1,52,10,5.06,51,2.80
7,malawi_2023,2,52,10,5.33,51,2.51
8,malawi_2023,3,52,12,5.50,48,2.69
9,malawi_2023,4,52,12,5.87,52,2.93


### 3c. Month by fold: how many fields have a clear photo

One table per cohort. A row is one fold, a column is one month, and a cell is the number of fields of that fold with a clear photo in that month.

A column of zeros is a month with **no photo in the catalogue** (`no_scene` in 3d), not a cloudy month:
Kenya 2019-02, and Oct + Nov 2022 for Malawi 2023 and Zimbabwe Lacuna. Together these are the 6,048 empty field-months.
More tries per month (data v2) cannot fill them. Only the older photo collection can.

In [20]:
cohorts = sql(f"SELECT DISTINCT cohort FROM {SPLIT} ORDER BY 1").cohort
for cohort in cohorts:
    print(f"=== {cohort}: clear fields per month and fold ===")
    display(sql(f"""
        PIVOT (SELECT s.fold, c.month, CASE WHEN c.state = 'clear' THEN 1 END AS clear
               FROM coverage c
               JOIN {SPLIT} s USING (obs_id)
               WHERE s.cohort = '{cohort}')
        ON month USING count(clear)
        GROUP BY fold
        ORDER BY fold
    """))

=== kenya_oaf_2019_2020: clear fields per month and fold ===


,fold,2019-02,2019-03,2019-04,2019-05,2019-06,2019-07,2019-08,2019-09,2020-02,2020-03,2020-04,2020-05,2020-06,2020-07,2020-08,2020-09
0,0,0,366,364,366,335,366,365,366,430,419,424,430,422,403,399,422
1,1,0,345,345,343,304,345,344,345,449,445,446,451,451,445,382,448
2,2,0,364,363,361,324,364,362,364,431,429,431,430,426,421,394,431
3,3,0,361,355,361,330,361,357,358,433,430,432,434,432,414,409,425
4,4,0,332,331,332,316,330,330,332,461,459,461,463,454,449,424,456


=== malawi_2023: clear fields per month and fold ===


,fold,2022-10,2022-11,2022-12,2023-01,2023-02,2023-03,2023-04,2023-05,2023-06
0,0,0,0,19,25,42,23,43,52,52
1,1,0,0,13,38,45,15,48,52,52
2,2,0,0,25,41,42,23,42,52,52
3,3,0,0,28,37,42,29,46,52,52
4,4,0,0,31,38,39,42,51,52,52


=== zimbabwe_2024: clear fields per month and fold ===


,fold,2023-10,2023-11,2023-12,2024-01,2024-02,2024-03,2024-04,2024-05,2024-06
0,0,492,489,486,465,489,484,490,492,492
1,1,492,492,476,453,492,485,492,492,492
2,2,491,491,485,465,492,487,492,492,492
3,3,491,490,490,452,490,488,491,491,491
4,4,491,491,490,461,489,482,491,491,491


=== zimbabwe_lacuna_2022_2023: clear fields per month and fold ===


,fold,2022-10,2022-11,2022-12,2023-01,2023-02,2023-03,2023-04,2023-05,2023-06
0,0,0,0,370,325,283,340,376,376,376
1,1,0,0,370,311,285,336,376,376,376
2,2,0,0,345,299,270,311,376,376,376
3,3,0,0,354,295,257,347,376,376,376
4,4,0,0,352,272,295,353,376,376,376


### 3d. Month by state: clear, cloudy or no photo (all folds together)

- `clear`: a photo with at least 80 % clear field pixels was found.
- `cloudy`: photos were tried, but every one had clouds or missing pixels over the field.
- `no_scene`: the photo catalogue had no photo of that month under the tile cloud limit (80 %).

In [21]:
for cohort in cohorts:
    print(f"=== {cohort}: fields per month and state ===")
    display(sql(f"""
        PIVOT (SELECT c.state, c.month
               FROM coverage c
               JOIN {SPLIT} s USING (obs_id)
               WHERE s.cohort = '{cohort}')
        ON month USING count(*)
        GROUP BY state
        ORDER BY state
    """))

=== kenya_oaf_2019_2020: fields per month and state ===


,state,2019-02,2019-03,2019-04,2019-05,2019-06,2019-07,2019-08,2019-09,2020-02,2020-03,2020-04,2020-05,2020-06,2020-07,2020-08,2020-09
0,clear,0,1768,1758,1763,1609,1766,1758,1765,2204,2182,2194,2208,2185,2132,2008,2182
1,cloudy,0,0,10,5,159,2,10,3,5,27,15,1,24,77,201,27
2,no_scene,1768,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


=== malawi_2023: fields per month and state ===


,state,2022-10,2022-11,2022-12,2023-01,2023-02,2023-03,2023-04,2023-05,2023-06
0,clear,0,0,116,179,210,132,230,260,260
1,cloudy,0,0,144,81,50,128,30,0,0
2,no_scene,260,260,0,0,0,0,0,0,0


=== zimbabwe_2024: fields per month and state ===


,state,2023-10,2023-11,2023-12,2024-01,2024-02,2024-03,2024-04,2024-05,2024-06
0,clear,2457,2453,2427,2296,2452,2426,2456,2458,2458
1,cloudy,1,5,31,162,6,32,2,0,0


=== zimbabwe_lacuna_2022_2023: fields per month and state ===


,state,2022-10,2022-11,2022-12,2023-01,2023-02,2023-03,2023-04,2023-05,2023-06
0,clear,0,0,1791,1502,1390,1687,1880,1880,1880
1,cloudy,0,0,89,378,490,193,0,0,0
2,no_scene,1880,1880,0,0,0,0,0,0,0


In [22]:
con.close()
print("closed")

closed
